# 06 — Publish the checkpoint, and pin it

**Gate this notebook closes: G6.**

Two artefacts leave here: `invasive_tile_v1_<init>.pt` and its `manifest.json`. The manifest
is not documentation — it is the input contract, the metrics, the provenance and a set of
**probe logits**, and step 8 refuses to load a checkpoint whose SHA-256 disagrees with it.

**The probe check is the most valuable test in the whole plan.** A fresh process reloads the
checkpoint, runs twenty named tiles, and has to reproduce the logits recorded at publish
time. It is the only test that catches a normalisation mismatch between training and serving
— a fault invisible to every metric computed inside the training process, because inside that
process both sides use the same wrong transform.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

In [ ]:
import json

import numpy as np
import torch

import bcss, datasets, export, hchannel, models, report

INIT = "imagenet"        # or "simclr" - whichever won the approach 3 A/B
INVERT_POLARITY = False
NAME = f"invasive_tile_v1_{INIT}"
TILE_PX, MPP = 224, 0.5

result = json.loads((backend_path.REPORTS_DIR / f"04_report_{INIT}.json").read_text())
print(f"publishing {NAME}")
print(f"  dice invasive vs non-invasive: "
      f"{result['held_out']['dice_invasive_vs_non_invasive']:.3f}")

## Assemble the full network

The head was fitted on cached features; the served model is the frozen body with that head
on top. Assembling it here rather than re-fitting keeps the published weights identical to
the ones that produced the report.

In [ ]:
net = models.resnet18_backbone(INIT, weights_dir=backend_path.PRETRAINED_DIR)
models.freeze_body(net)

head_state = torch.load(backend_path.REPORTS_DIR / f"04_head_{INIT}.pt", weights_only=True)
net.fc.load_state_dict(head_state)
net.eval()

print(f"assembled: resnet18({INIT}) + Linear(512, {len(bcss.CLASS_NAMES)})")
print(f"classes: {list(bcss.CLASS_NAMES)}")

## The probe tiles

Twenty tiles from the held-out institutions, named in the manifest with the logits this model
produced for them.

In [ ]:
rows = datasets.read_manifest(backend_path.TILES_DIR / "tiles_manifest.csv")
_, test_rows = datasets.institution_split(rows)

rng = np.random.default_rng(0)
# Stratified, so the probe covers all three classes rather than twenty tiles of stroma.
probe_rows = []
for cls in range(len(bcss.CLASS_NAMES)):
    pool = [r for r in test_rows if int(r["label"]) == cls]
    picks = rng.choice(len(pool), size=min(7, len(pool)), replace=False)
    probe_rows.extend(pool[int(i)] for i in picks)

probe_set = datasets.TileDataset(probe_rows, backend_path.TILES_DIR,
                                 augment=False, invert=INVERT_POLARITY)
with torch.inference_mode():
    batch = torch.stack([probe_set[i][0] for i in range(len(probe_set))])
    logits = net(batch).numpy()

probe_tiles = [
    {"tile_id": row["tile_id"], "tile_path": row["tile_path"], "label": int(row["label"]),
     "logits": [round(float(v), 6) for v in logits[i]]}
    for i, row in enumerate(probe_rows)
]
print(f"{len(probe_tiles)} probe tiles, "
      f"{sum(1 for p in probe_tiles if np.argmax(p['logits']) == p['label'])} predicted correctly")

In [ ]:
summary = json.loads((backend_path.TILES_DIR / "export_summary.json").read_text())
download = json.loads((backend_path.BCSS_DIR / "download_manifest.json").read_text())

checkpoint, manifest_path = models.publish(
    net,
    name=NAME,
    models_dir=backend_path.MODELS_DIR,
    init=INIT,
    invert_polarity=INVERT_POLARITY,
    tile_px=TILE_PX,
    mpp=MPP,
    metrics=result["held_out"] | {"groupkfold_dice": result["groupkfold_dice"]},
    training_data={
        "source": "BCSS (Amgad et al. 2019), CC0 1.0",
        "regions_total": download["summary"]["regions"],
        "regions_trained_on": download["summary"]["train_regions"],
        "tiles": summary["tiles_kept"],
        "per_class": summary["per_class"],
        "pseudo_labels": False,
        "note": ("AICAN pseudo-labels (input B2) deferred: no H&E slide on disk. "
                 "BCSS-only is a complete model; see approach-1-training-plan.md Phase 5."),
    },
    provenance={
        "exporter_spec": summary["spec"],
        "tiles_manifest_sha256": models.sha256_of(
            backend_path.TILES_DIR / "tiles_manifest.csv"),
        "pretrained_sha256": models.sha256_of(
            backend_path.PRETRAINED_DIR / models.INITS[INIT]),
        "licences": {
            "BCSS": "CC0 1.0", "torchvision resnet18": "BSD-3",
            **({"ciga simclr resnet18": "MIT"} if INIT == "simclr" else {}),
        },
    },
    probe_tiles=probe_tiles,
)

print(f"{checkpoint.name}  {checkpoint.stat().st_size / 1e6:.1f} MB")
print(f"{manifest_path.name}")
print(f"\nSHA-256: {json.loads(manifest_path.read_text())['sha256']}")
print("\nRecord that hash in models/README.md. Nothing loads this file without checking it.")

## G6 — reload in a *fresh* process

Run in a subprocess, not in this kernel. A check that runs in the process that just trained
the model shares its imports, its module state and its transform — which is exactly what
makes it unable to catch the fault it is meant to catch.

In [ ]:
import subprocess, sys, textwrap

check = textwrap.dedent(f'''
    import sys, json, pathlib
    import numpy as np, torch
    sys.path.insert(0, r"{backend_path.PROJECT_ROOT / 'src'}")
    import backend_path, datasets, models

    net, manifest = models.load_pinned("{NAME}", models_dir=backend_path.MODELS_DIR)
    probes = manifest["probe_tiles"]

    dataset = datasets.TileDataset(
        [{{"tile_path": p["tile_path"], "label": p["label"], "tile_id": p["tile_id"]}}
         for p in probes],
        backend_path.TILES_DIR, augment=False,
        invert=manifest["input"]["invert_polarity"],
    )
    with torch.inference_mode():
        logits = net(torch.stack([dataset[i][0] for i in range(len(dataset))])).numpy()

    expected = np.array([p["logits"] for p in probes])
    worst = float(np.max(np.abs(logits - expected)))
    print(f"worst logit difference: {{worst:.3e}}")
    assert worst < 1e-4, (
        "a freshly loaded checkpoint does not reproduce its own recorded logits. "
        "The input transform at load time differs from the one at training time - "
        "which is the exact failure this check exists for."
    )
    print("G6 OK - the published model reproduces its manifest in a fresh process")
''')

completed = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(completed.stdout or completed.stderr)
assert completed.returncode == 0, "G6 failed - do not deploy this checkpoint"

---

- [ ] G6a — checkpoint and manifest written, SHA-256 recorded in `models/README.md`
- [ ] G6b — a fresh process reloads it and reproduces the probe logits
- [ ] G6c — `models/.gitignore` still excludes the binary

## Handing it to step 8

`backend/app/pipeline/step08_tissue_type_segmentation/pipeline.py` currently raises
`StepNotImplementedError`. To finish:

1. `settings.tissue_model_dir` / `tissue_model_name`, following the `qc_models_dir` pattern.
2. A loader calling `models.load_pinned` — which **refuses** a checkpoint whose hash does not
   match its manifest, rather than silently scoring with unknown weights.
3. `scripts/check_tissue_model.py`, beside the existing `check_qc_models.py`.
4. No grid of its own: with `settings.tile_size = 224`, step 7's kept tiles *are* the model's
   inputs — one tile, one forward pass, no second geometry anywhere in the codebase.
5. `app/data/pipeline_steps.py`: `implemented=True` for `tissue-type-segmentation`.

**Gate G7:** the same slide tile, scored through this notebook and through the API, returns
the same class. Until that passes the model is trained, not deployed.